# Fetching Census Survey Data

morpc-census connects to the US Census Bureau API, retrieves survey data, and returns it as a normalized long-format DataFrame ready for analysis. Every request is defined by:

- an **Endpoint** — survey dataset and vintage year
- a **scope** — geographic extent (a region, county, state, etc.)
- a **Group** or a list of **variables** — what to retrieve
- an optional **sumlevel** — resolution within the scope (county, tract, place, etc.)

This notebook walks through a typical workflow from discovery to analysis to saving output.

In [12]:
from morpc.logs import config_logs

config_logs('morpc-census-demo.log', 'debug')

2026-05-12 08:54:19,671 | INFO | morpc.logs.config_logs: Set up logging save to file "morpc-census-demo.log", log level "debug"


In [13]:
from morpc_census import (
    Endpoint, Group, CensusAPI, DimensionTable,
    get_all_avail_endpoints,
    IMPLEMENTED_ENDPOINTS, SCOPES,
)
import pandas as pd

## 1. Endpoints — choosing a survey and year

`IMPLEMENTED_ENDPOINTS` lists every survey/table the package supports. `Endpoint` validates the survey name and vintage year at construction time, so you find out immediately if you've specified something unavailable.

In [14]:
# Surveys the package supports
IMPLEMENTED_ENDPOINTS

['acs/acs1',
 'acs/acs1/profile',
 'acs/acs1/subject',
 'acs/acs5',
 'acs/acs5/profile',
 'acs/acs5/subject',
 'dec/pl',
 'dec/dhc',
 'dec/ddhca',
 'dec/ddhcb',
 'dec/sf1',
 'dec/sf2',
 'dec/sf3',
 'geoinfo']

In [15]:
get_all_avail_endpoints()

{'abscb': [2017, 2018, 2019, 2020, 2021, 2022, 2023],
 'abscbo': [2017, 2018, 2019, 2020, 2021, 2022, 2023],
 'abscs': [2017, 2018, 2019, 2020, 2021, 2022, 2023],
 'absmcb': [2020, 2021, 2022, 2023],
 'absnesd': [2018, 2019, 2020, 2021, 2022, 2023],
 'absnesdo': [2018, 2019, 2020, 2021, 2022, 2023],
 'abstcb': [2018],
 'acs/acs1': [2010,
  2011,
  2012,
  2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2005,
  2006,
  2007,
  2008,
  2009,
  2019,
  2021,
  2022,
  2023,
  2024],
 'acs/acs1/cprofile': [2010,
  2011,
  2012,
  2019,
  2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2021,
  2022,
  2023,
  2024],
 'acs/acs1/profile': [2010,
  2011,
  2012,
  2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2005,
  2006,
  2007,
  2008,
  2009,
  2019,
  2021,
  2022,
  2023,
  2024],
 'acs/acs1/pums': [2004,
  2005,
  2006,
  2007,
  2008,
  2009,
  2010,
  2011,
  2012,
  2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2019,
  2021,
  2022,
  2023,
  2024],
 'acs/acs1/pumspr': [2005,
  2

In [16]:
# Construct an endpoint — raises ValueError for unknown surveys or unavailable years
ep = Endpoint('acs/acs5', 2023)
ep

Endpoint('acs/acs5', 2023)

In [17]:
# All vintage years available for this survey
ep.vintages

[2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2009,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024]

## 2. Groups — browsing available variables

> **Network required** — the cells below make live calls to the Census API.

A *group* is a table within a survey — a collection of related variables. `endpoint.groups` returns every group with its description. Fetched once, then cached on the object.

In [18]:
# Browse available groups — descriptions keyed by group code
{k: v['description'] for k, v in list(ep.groups.items())[:10]}

2026-05-12 08:54:24,777 | DEBUG | morpc_census.api.groups: Fetching groups for 2023 acs/acs5
2026-05-12 08:54:24,780 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2023/acs/acs5/groups.json with parameters {'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 08:54:24,782 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443


2026-05-12 08:54:25,064 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2023/acs/acs5/groups.json?key=83269ff2739cb3bd485c75b091dcee493ad6fe70 HTTP/1.1" 200 None
2026-05-12 08:54:25,496 | DEBUG | morpc.req.get_json_safely: Request successful. Decoding return JSON.


{'B01001': 'Sex by Age',
 'B01001A': 'Sex by Age (White Alone)',
 'B01001B': 'Sex by Age (Black or African American Alone)',
 'B01001C': 'Sex by Age (American Indian and Alaska Native Alone)',
 'B01001D': 'Sex by Age (Asian Alone)',
 'B01001E': 'Sex by Age (Native Hawaiian and Other Pacific Islander Alone)',
 'B01001F': 'Sex by Age (Some Other Race Alone)',
 'B01001G': 'Sex by Age (Two or More Races)',
 'B01001H': 'Sex by Age (White Alone, Not Hispanic or Latino)',
 'B01001I': 'Sex by Age (Hispanic or Latino)'}

In [19]:
# Construct a Group — validates the code against endpoint.groups
group = Group(ep, 'B01001')
print(group.description)
print(group.universe)

Sex by Age
Total population


In [20]:
# Variable codes and labels for this group
{k: v.get('label', '') for k, v in list(group.variables.items())[:8]}

2026-05-12 08:54:29,124 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2023/acs/acs5/groups/B01001.json with parameters {'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 08:54:29,126 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443


2026-05-12 08:54:29,334 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2023/acs/acs5/groups/B01001.json?key=83269ff2739cb3bd485c75b091dcee493ad6fe70 HTTP/1.1" 200 19404
2026-05-12 08:54:29,367 | DEBUG | morpc.req.get_json_safely: Request successful. Decoding return JSON.


{'B01001_001E': 'Estimate!!Total:',
 'B01001_001EA': 'Annotation of Estimate!!Total:',
 'B01001_001M': 'Margin of Error!!Total:',
 'B01001_001MA': 'Annotation of Margin of Error!!Total:',
 'B01001_002E': 'Estimate!!Total:!!Male:',
 'B01001_002EA': 'Annotation of Estimate!!Total:!!Male:',
 'B01001_002M': 'Margin of Error!!Total:!!Male:',
 'B01001_002MA': 'Annotation of Margin of Error!!Total:!!Male:'}

## 3. Scopes — choosing a geographic extent

A scope names the geographic coverage of the request — which places to include. `SCOPES` lists all built-in scopes. Pass a scope key to `CensusAPI` as a string.

In [21]:
# Available scope keys
list(SCOPES.keys())

['us',
 'columbuscbsa',
 'alabama',
 'alaska',
 'arizona',
 'arkansas',
 'california',
 'colorado',
 'connecticut',
 'delaware',
 'district of columbia',
 'florida',
 'georgia',
 'hawaii',
 'idaho',
 'illinois',
 'indiana',
 'iowa',
 'kansas',
 'kentucky',
 'louisiana',
 'maine',
 'maryland',
 'massachusetts',
 'michigan',
 'minnesota',
 'mississippi',
 'missouri',
 'montana',
 'nebraska',
 'nevada',
 'new hampshire',
 'new jersey',
 'new mexico',
 'new york',
 'north carolina',
 'north dakota',
 'ohio',
 'oklahoma',
 'oregon',
 'pennsylvania',
 'rhode island',
 'south carolina',
 'south dakota',
 'tennessee',
 'texas',
 'utah',
 'vermont',
 'virginia',
 'washington',
 'west virginia',
 'wisconsin',
 'wyoming',
 'puerto rico',
 'lake',
 'hancock',
 'allen',
 'morgan',
 'portage',
 'butler',
 'fayette',
 'vinton',
 'paulding',
 'scioto',
 'fulton',
 'henry',
 'logan',
 'brown',
 'monroe',
 'trumbull',
 'pike',
 'pickaway',
 'muskingum',
 'lawrence',
 'adams',
 'crawford',
 'guernsey',
 

## 4. Fetching data

`CensusAPI` validates parameters, builds the request, fetches the data, and transforms it into a normalized long-format table in one step. `api.data` holds the raw wide response from the API; `api.long` is the normalized output.

In [22]:
# Sex by age for all counties in the 15-county MORPC region
b01001 = CensusAPI(ep, 'region15', group=group)
b01001.data

2026-05-12 08:54:41,178 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.__init__: Initializing CensusAPI for census-acs-acs5-2023-region15-b01001.
2026-05-12 08:54:41,179 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.__init__: Building request URL and parameters.
2026-05-12 08:54:41,180 | DEBUG | morpc_census.geos.geoids_from_scope: Fetching geoids from scope 'region15'.
2026-05-12 08:54:41,182 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2023/geoinfo?get=GEO_ID with parameters {'for': 'county:041,045,049,089,097,129,159,083,101,117,047,073,091,127,141', 'in': 'state:39', 'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 08:54:41,184 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443


2026-05-12 08:54:41,615 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2023/geoinfo?get=GEO_ID&for=county%3A041%2C045%2C049%2C089%2C097%2C129%2C159%2C083%2C101%2C117%2C047%2C073%2C091%2C127%2C141&in=state%3A39&key=83269ff2739cb3bd485c75b091dcee493ad6fe70 HTTP/1.1" 200 None
2026-05-12 08:54:41,616 | DEBUG | morpc.req.get_json_safely: Request successful. Decoding return JSON.
2026-05-12 08:54:41,620 | DEBUG | morpc_census.geos.geoinfo_from_scope_sumlevel: Building parameters for scope 'region15' at sumlevel None.
2026-05-12 08:54:41,621 | INFO | morpc_census.geos.geoinfo_from_scope_sumlevel: No sumlevel specified; using scope 'region15' parameters.
2026-05-12 08:54:41,622 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.__init__: Fetching data from https://api.census.gov/data/2023/acs/acs5? with params {'get': 'group(B01001)', 'for': 'county:041,045,049,089,097,129,159,083,101,117,047,073,091,127,141', 'in': 'state:39'}.
2026

,B01001_001E,B01001_001EA,B01001_001M,B01001_001MA,B01001_002E,B01001_002EA,B01001_002M,B01001_002MA,B01001_003E,B01001_003EA,...,B01001_048M,B01001_048MA,B01001_049E,B01001_049EA,B01001_049M,B01001_049MA,GEO_ID,NAME,state,county
0,221160,NaN,-555555555,*****,110668,NaN,33,NaN,6456,NaN,...,290,NaN,2009,NaN,352,NaN,0500000US39041,"Delaware County, Ohio",39,41
1,161289,NaN,-555555555,*****,80584,NaN,128,NaN,4690,NaN,...,241,NaN,1892,NaN,225,NaN,0500000US39045,"Fairfield County, Ohio",39,45
2,28880,NaN,-555555555,*****,14271,NaN,163,NaN,778,NaN,...,106,NaN,352,NaN,91,NaN,0500000US39047,"Fayette County, Ohio",39,47
3,1321635,NaN,-555555555,*****,649003,NaN,61,NaN,44950,NaN,...,692,NaN,11174,NaN,747,NaN,0500000US39049,"Franklin County, Ohio",39,49
4,27938,NaN,-555555555,*****,14022,NaN,110,NaN,808,NaN,...,102,NaN,248,NaN,87,NaN,0500000US39073,"Hocking County, Ohio",39,73
5,62888,NaN,-555555555,*****,31021,NaN,150,NaN,1943,NaN,...,175,NaN,895,NaN,160,NaN,0500000US39083,"Knox County, Ohio",39,83
6,180311,NaN,-555555555,*****,89196,NaN,116,NaN,5355,NaN,...,266,NaN,2124,NaN,256,NaN,0500000US39089,"Licking County, Ohio",39,89
7,46140,NaN,-555555555,*****,23206,NaN,100,NaN,1435,NaN,...,158,NaN,464,NaN,107,NaN,0500000US39091,"Logan County, Ohio",39,91
8,44126,NaN,-555555555,*****,24262,NaN,160,NaN,1181,NaN,...,126,NaN,525,NaN,130,NaN,0500000US39097,"Madison County, Ohio",39,97
9,65145,NaN,-555555555,*****,35012,NaN,168,NaN,1943,NaN,...,185,NaN,772,NaN,158,NaN,0500000US39101,"Marion County, Ohio",39,101


## 5. The long-format result

`api.long` is the normalized output: one row per geography × variable. Type suffixes are split into separate columns (`estimate`, `moe`), variable labels are extracted from the raw label strings, and Census missing-value codes are converted to `NaN`.

In [23]:
# Long-format output — one row per geography × variable
b01001.long

,geoidfq,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
0,0500000US39041,2023,acs/acs5,Sex by age,Total population,Total:,B01001_001,221160,-555555555
25,0500000US39041,2023,acs/acs5,Sex by age,Total population,Total:!!Male:,B01001_002,110668,33
48,0500000US39041,2023,acs/acs5,Sex by age,Total population,Total:!!Male:!!Under 5 years,B01001_003,6456,6
37,0500000US39041,2023,acs/acs5,Sex by age,Total population,Total:!!Male:!!5 to 9 years,B01001_004,7560,466
26,0500000US39041,2023,acs/acs5,Sex by age,Total population,Total:!!Male:!!10 to 14 years,B01001_005,9200,465
...,...,...,...,...,...,...,...,...,...
705,0500000US39159,2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!67 to 69 years,B01001_045,914,183
706,0500000US39159,2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!70 to 74 years,B01001_046,1201,211
707,0500000US39159,2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!75 to 79 years,B01001_047,738,161
708,0500000US39159,2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!80 to 84 years,B01001_048,571,129


In [24]:
# Column types in the long-format output
b01001.long.dtypes

geoidfq             object
reference_period     int64
survey              object
concept             object
universe            object
variable_label      object
variable            object
estimate             int64
moe                  int64
dtype: object

## 6. GEOIDFQs — parsing geography identifiers

The `GEOIDFQ` column holds fully-qualified geography identifiers (e.g. `0500000US39049`). `api.geoidfqs` parses each one into a `GeoIDFQ` object with typed fields for summary level, variant, and component codes.

In [25]:
# First three geography IDs parsed as GeoIDFQ objects
b01001.geoidfqs[:3]

[GeoIDFQ(sumlevel='050', variant='00', geocomp='00', state='39', county='041'),
 GeoIDFQ(sumlevel='050', variant='00', geocomp='00', state='39', county='045'),
 GeoIDFQ(sumlevel='050', variant='00', geocomp='00', state='39', county='047')]

In [26]:
# Fields on a GeoIDFQ object
g = b01001.geoidfqs[0]
print('summary level:', g.sumlevel)
print('geoid:        ', g.geoid)
print('parts:        ', g.parts)
print('string form:  ', str(g))

summary level: '050'
geoid:         39041
parts:         {'state': '39', 'county': '041'}
string form:   0500000US39041


## 7. Drilling down with sumlevel

The `sumlevel` parameter fetches child geographies within the scope. For example, `sumlevel='tract'` with `scope='franklin'` returns all census tracts in Franklin County.

In [27]:
# Place of birth for all tracts in Franklin County
b05006_tracts = CensusAPI(ep, 'franklin', group=Group(ep, 'B05006'), sumlevel='tract')
b05006_tracts.long

2026-05-12 08:54:54,860 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-countytract-franklin-b05006.__init__: Initializing CensusAPI for census-acs-acs5-2023-countytract-franklin-b05006.
2026-05-12 08:54:54,864 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-countytract-franklin-b05006.__init__: Building request URL and parameters.
2026-05-12 08:54:54,865 | DEBUG | morpc_census.geos.geoids_from_scope: Fetching geoids from scope 'franklin'.
2026-05-12 08:54:54,866 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2023/geoinfo?get=GEO_ID with parameters {'for': 'county:049', 'in': 'state:39', 'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 08:54:54,868 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443
2026-05-12 08:54:55,348 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2023/geoinfo?get=GEO_ID&for=county%3A049&in=state%3A39&key=83269ff2

,geoidfq,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
0,1400000US39049000110,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:,B05006_001,60,37
125,1400000US39049000110,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:,B05006_002,17,19
145,1400000US39049000110,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:!!Northern Europe:,B05006_003,12,18
146,1400000US39049000110,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:!!Northern Europe:!!Denmark,B05006_004,0,13
147,1400000US39049000110,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:!!Northern Europe:!!Ireland,B05006_005,0,13
...,...,...,...,...,...,...,...,...,...
58279,1400000US39049980000,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Latin America:!!South Ameri...,B05006_174,0,13
58276,1400000US39049980000,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Latin America:!!South Ameri...,B05006_175,0,13
58280,1400000US39049980000,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Northern America:,B05006_176,0,13
58281,1400000US39049980000,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Northern America:!!Canada,B05006_177,0,13


## 8. Fetching specific variables without a group

When you only need a few variables — possibly from different groups — pass `variables` directly instead of a group. The Census API allows at most 50 fields per request; the package batches automatically when you exceed that limit.

In [ ]:
# Total population, male, and female — fetched directly without a group
pop = CensusAPI(ep, 'region15', variables=['B01001_001E', 'B01001_002E', 'B01001_026E'])
pop.long

## 9. Analyzing with DimensionTable

`DimensionTable` parses each variable's `!!`-delimited label into named **dimension columns** and pivots the long-format data into a readable wide table.

Each label segment is classified as a *subtotal* (ends with `:`) or a *leaf* (no trailing `:`). Subtotals are left-aligned into the first columns; leaves into the last. This alignment keeps the same concept in the same column regardless of path depth — for example, in B05004 (Nativity and Citizenship × Sex), *Male* and *Female* always land in the last column whether there are one or two nativity levels before them.

For B01001 (Sex by Age), `Total:!!Male:!!Under 5 years` parses as:

| dim | value |
|-----|-------|
| `total` | `Total:` |
| `sex` | `Male:` |
| `age` | `Under 5 years` |

Key methods:
- `drop(dim, method='summarize')` — remove a dimension level before pivoting
- `remap(variable_map)` — relabel and collapse variables, then re-parse dimensions
- `wide()` — produce the wide pivot table
- `percent()` — column percentages relative to the grand total row

In [ ]:
# Construct a DimensionTable; dim_names labels the parsed dimension columns
dim = DimensionTable(b01001.long, dim_names=['total', 'sex', 'age'])

# Each row is one Census variable; each column is a level of the label hierarchy
# ':' suffix is preserved here — it marks subtotals (rows that have sub-dimensions)
dim.dims.head(8)

In [ ]:
# Full wide table — all dimension levels as row index, geography × value type as columns
# ':' is stripped from displayed index values (Total: → Total, Male: → Male, etc.)
dim.wide()

In [ ]:
# Drop the age dimension: keep only the pre-computed sex-level rows
# (grand total + Male total + Female total — one row per sex per county)
dim.drop('age').wide()

In [ ]:
from morpc_census import AGEGROUP_MAP

# B01001 uses uneven age bins: "15 to 17 years" and "18 and 19 years" are separate
# variables, as are "20 years", "21 years", and "22 to 24 years".
# AGEGROUP_MAP collapses these into standard 5-year groups. remap() applies the
# substitutions and sums estimates for any rows that share a new label.
dim_remapped = DimensionTable(b01001.long, dim_names=['total', 'sex', 'age'])
dim_remapped.remap(AGEGROUP_MAP)

# Population by standardized age group — aggregate sex out (sum Male + Female)
dim_remapped.drop('sex', method='aggregate').wide()

In [ ]:
# Sex percentages relative to the county total (age dropped; grand total = 100%)
dim.drop('age').percent()

## 10. Time series

`DimensionTable` accepts any long-format DataFrame with the standard schema. Concatenating `long` from multiple vintages produces a time-series table where `reference_period` becomes a column dimension.

In [ ]:
# Fetch the same group for an earlier vintage
ep2018 = Endpoint('acs/acs5', 2018)
b01001_2018 = CensusAPI(ep2018, 'region15', group=Group(ep2018, 'B01001'))

# Stack the two years and build a dimension table
long_ts = pd.concat([b01001_2018.long, b01001.long])
DimensionTable(long_ts).wide()

## 11. Saving output

`api.save()` writes three files to the output directory:

- `{name}.long.csv` — the long-format data
- `{name}.schema.yaml` — a frictionless Schema describing every column
- `{name}.resource.yaml` — a frictionless Resource descriptor linking the CSV to its schema

The resource is validated immediately after writing, so any schema mismatch surfaces here.

In [ ]:
import os

b01001.save('./temp_data')

print('filename:', b01001.filename)
print('exists:  ', os.path.exists(f'./temp_data/{b01001.filename}'))

In [ ]:
from frictionless import Resource

Resource(f'temp_data/{b01001.name}.resource.yaml')

In [ ]:
from frictionless import Schema

Schema(f'temp_data/{b01001.name}.schema.yaml')